# RAG with LlamIndex - Exp2

* Explore free or cheaper LLMs --> RAG pipeline can switch LLMs

In [1]:
%load_ext autoreload
%autoreload 2

import os
os.environ.pop("OPENAI_API_KEY", None)
from groq import Groq

from datasets import load_dataset
import pandas as pd
import json
from llama_index.core import (
    VectorStoreIndex,
    Settings,
)

from utils import *

import warnings
warnings.filterwarnings('ignore')

resource module not available on Windows


### Load Small Testing Data

In [2]:
dataset = load_dataset("PatronusAI/financebench")

selected_docs =set()
for dct in dataset['train'].select(range(5)):
    print(dct['doc_name'])
    print(dct['question'])
    selected_docs.add(f'pdfs/{dct["doc_name"]}.pdf')
    print('--------------')
print(len(selected_docs))

3M_2018_10K
What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.
--------------
3M_2018_10K
Assume that you are a public equities analyst. Answer the following question by primarily using information that is shown in the balance sheet: what is the year end FY2018 net PPNE for 3M? Answer in USD billions.
--------------
3M_2022_10K
Is 3M a capital-intensive business based on FY2022 data?
--------------
3M_2022_10K
What drove operating margin change as of FY2022 for 3M? If operating margin is not a useful metric for a company like this, then please state that and explain why.
--------------
3M_2022_10K
If we exclude the impact of M&A, which segment has dragged down 3M's overall growth in 2022?
--------------
2


In [3]:
# load selected pdfs
loaded_pdf = {}
with ThreadPoolExecutor(max_workers=5) as executor:
    loaded_pdf = dict(executor.map(load_pdf_content, list(selected_docs)))

print(f'{len(loaded_pdf)} pdfs loaded')

2 pdfs loaded


In [4]:
# Genrate LlamaIndex Documents from each selected item
documents = []
selected_metadata_cols = ['company', 'doc_name', 'doc_period', 'doc_link']

documents = process_items_parallel(
    dataset['train'],
    selected_doc_names=set([selected_doc.replace('pdfs/', '').replace('.pdf', '')
                             for selected_doc in selected_docs]),
    loaded_pdf=loaded_pdf,
    selected_metadata_cols=selected_metadata_cols,
    max_workers=10
)

print(f"Generated {len(documents)} documents")

Generated 5 documents


### RAG Pipeline

In [6]:
import os
from llama_index.core import Settings
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import StorageContext, load_index_from_storage

os.environ["GROQ_API_KEY"] = os.environ["GROQ_TOKEN"]
Settings.llm = Groq(
    model="llama-3.1-8b-instant",
    temperature=0,
)

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5", 
    device="cpu"
)
node_parser = setup_chunking_strategy(embed_model=Settings.embed_model)

print("Model name:", getattr(Settings.llm, "model", None))
print("Model name:", getattr(Settings.embed_model, "model_name",
                              getattr(Settings.embed_model, "model_id", None)))

Model name: llama-3.1-8b-instant
Model name: BAAI/bge-small-en-v1.5


In [7]:
if os.path.isdir("./storage"):
    storage_context = StorageContext.from_defaults(persist_dir="./storage")
    vector_index = load_index_from_storage(storage_context)
else:
    vector_index = VectorStoreIndex.from_documents(
        documents,
        node_parser=node_parser,
        show_progress=True
    )
    vector_index.storage_context.persist(persist_dir="./storage")
    print("Index saved to ./storage")

Parsing nodes:   0%|          | 0/5 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/1191 [00:00<?, ?it/s]

Index saved to ./storage


In [8]:
retriever = HybridRetriever(
            vector_index,
            documents,
            top_k=1,
            alpha=0.6  # 60% weight to semantic retrieval (dense), 40% to key words retrieval (sparse)
        )
query_engine = get_query_engine(retriever, reranker=None)  # without reranker
query_engine

In [10]:
selected_items = dataset['train'].select(range(5))
eval_lst = await run_eval_async(selected_items, query_engine, concurrency=5)
print(len(eval_lst))

with open("output/exp2_raw.json", "w", encoding="utf-8") as f:
    json.dump(eval_lst, f, ensure_ascii=False, indent=2)

5
